In [ ]:
from dotenv import load_dotenv
load_dotenv()
import uuid
from typing import List
from pydantic import BaseModel,Field

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph,START, END,MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore


In [ ]:
store = InMemoryStore()

In [ ]:
extractor_llm = ChatGroq(model="openai/gpt-oss-120b",max_tokens=600,temperature=0)

In [ ]:
class MemoryItem(BaseModel):
    text: str=Field(description="Atomic user as a short sentence")
    is_new: bool = Field(description="True if this memory is new and should be stored. False if duplicates/already known")

In [ ]:
class MemoryDecision(BaseModel):
    should_write: bool=Field(description="Whether to store any memories")
    memories: List[MemoryItem] = Field(default_factory=list, description="Atomic user memories to store")

In [ ]:
memory_extractor = extractor_llm.with_structured_output(MemoryDecision)

In [ ]:
MEMORY_PROMPT= """
Your are responsible for updating and maintaining accurate user memories.
CURRENT USER DETAILS (Existing memories):
{user_details_content}

TASK:
- Review the user's latest message.
- Extract user-specifc info worth storing long0term (idetnity, stable preferences,ongoinf projeccts/goals).
- For each extracted item, set is_new=True Only if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false
- Keep each memory as a short by the user.
- If there is nothing memory-worthy, return an empty list.
"""

In [ ]:
def chat_create_memory_node(state: MessagesState,config: RunnableConfig, store: BaseStore):
    user_id=config["configurable"]["user_id"]

    namespace=("user",user_id,"details")

    existing_items = store.search(namespace)
    existing_texts = [it.value.get("data","") for it in existing_items if it.value.get("data")]
    user_details_content = "\n".join(f"-{t}" for t in existing_texts) if existing_texts else "(empty)"


    last_text = state["messages"][-1]

    decision: MemoryDecision = memory_extractor.invoke(
        [
            SystemMessage(content=MEMORY_PROMPT.format(user_details_content=user_details_content)),
                                   {"role":"user","content":f"USER MESSAGE:\n{last_text}"},
        ]
    )

    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new:
                store.put(namespace,str(uuid.uuid4()),{"data":mem.text})

    return {"messages":[{"role":"assistant","content":"Noted.."}]}

In [ ]:
builder=StateGraph(MessagesState)
builder.add_node("remember",remember_only_node)
builder.add_edge(START,"remember")
builder.add_edge("remember",END)

graph = builder.compile(store=store)
graph

In [ ]:
config={"configurable":{"user_id":"u1"}}

res1=graph.invoke({"messages":[{"role":"user","content":"Hi my name is Maaz"}]},config)
print(res["messages"][-1].content)

In [ ]:
res2=graph.invoke({"messages":[{"role":"user","content":"I am fresher looking for job"}]},config)
print(res2["messages"][-1].content)

In [ ]:
res3=graph.invoke({"messages":[{"role":"user","content":"I am skilled in python"}]},config)
print(res3["messages"][-1].content)

In [ ]:
items = store.search(("user","u1","details"))
for i in items:
    print(i.value["data"])